# Graphs for the website

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, HeatMap
import json

In [3]:
# Load the data
df_map = pd.read_csv("sf_movies_cleaned_og.csv")

In [6]:
# Get number of unique titles with values for box office and number of does who have no box office
df_titles = df_map.drop_duplicates(subset=["Title"])

has_bo    = df_titles["Box_office"].notna().sum()
no_bo     = df_titles["Box_office"].isna().sum()

print(f"Titles with box office:    {has_bo}")
print(f"Titles without box office: {no_bo}")
print(f"Total unique titles:       {has_bo + no_bo}")

# Check if all Titles with box office are movies (not TV shows)
movies_with_bo = df_titles[df_titles["Box_office"].notna()]["Kind"].value_counts().get("movie", 0)
print(f"Movies with box office:    {movies_with_bo}")


Titles with box office:    202
Titles without box office: 114
Total unique titles:       316
Movies with box office:    202


### Interactive Map of Filming locations in San Francisco

In [8]:
# Drop rows with missing latitude or longitude
df_map = df_map.dropna(subset=["Latitude", "Longitude"]).copy()

# Extract the films genre for colors
df_map["Primary_Genre"] = (
    df_map["Genres"]
    .fillna("Other")
    .str.split(",")
    .str[0]
    .str.strip()
)

# Colors for mapping genres
GENRE_COLORS = {
    "Drama":        "#185FA5",
    "Thriller":     "#993C1D",
    "Crime":        "#533AB7",
    "Comedy":       "#0F6E56",
    "Action":       "#BA7517",
    "Romance":      "#993556",
    "Documentary":  "#3B6D11",
    "Horror":       "#3C3489",
    "Sci-Fi":       "#5F5E5A",
    "Biography":    "#0F6E56",
    "Mystery":      "#533AB7",
    "Other":        "#888780",
}

# Function to get color based on genre
def get_color(genre):
    for key in GENRE_COLORS:
        if key.lower() in str(genre).lower():
            return GENRE_COLORS[key]
    return GENRE_COLORS["Other"]

# HTML popup for the points
def build_popup(row):
    poster_html = (
        f'<img src="{row["Poster"]}" '
        f'style="width:90px;min-height:120px;object-fit:cover;flex-shrink:0;">'
        if pd.notna(row.get("Poster")) and row.get("Poster") not in ["N/A", ""]
        else f'<div style="width:90px;min-height:120px;background:#e8e8e4;flex-shrink:0;'
             f'display:flex;align-items:center;justify-content:center;'
             f'font-size:22px;color:#aaa;">🎬</div>'
    )

    rating_str = f"{row['Imdb_rating']:.1f}" if pd.notna(row.get("Imdb_rating")) else "N/A"
    year_str   = str(int(row["Year"])) if pd.notna(row.get("Year")) else "N/A"
    genre_str  = row.get("Primary_Genre", "N/A")
    color      = get_color(genre_str)

    # Lighten the genre color for the badge background (add alpha via rgba workaround)
    location_str      = row.get("Locations", "N/A")
    neighborhood_str  = row.get("Analysis neighborhood", "N/A")
    director_str      = row.get("Director", "N/A")
    boxing_str        = row.get("Box_office", "N/A")
    sentiment_raw     = row.get("Sentiment", "N/A")

    # Sentiment dot color
    sent_color = "#1D9E75" if "positive" in str(sentiment_raw).lower() else \
                 "#D85A30" if "negative" in str(sentiment_raw).lower() else "#888780"

    html = f"""
    <div style="
        font-family: sans-serif;
        font-size: 12px;
        width: 240px;
        background: #fff;
        border-radius: 10px;
        overflow: hidden;
        line-height: 1.5;
        box-shadow: 0 2px 12px rgba(0,0,0,0.12);
    ">
        <!-- Header: poster + title block side-by-side -->
        <div style="display:flex;align-items:stretch;">
            {poster_html}
            <div style="padding:10px 12px;display:flex;flex-direction:column;
                        justify-content:center;min-width:0;gap:4px;">
                <div style="font-size:14px;font-weight:600;color:#1a1a1a;
                            line-height:1.3;">{row['Title']}</div>
                <div style="font-size:11px;color:#888;">{year_str}</div>
                <div style="display:inline-flex;width:fit-content;
                            background:{color}22;border-radius:4px;
                            padding:2px 7px;font-size:11px;
                            color:{color};font-weight:500;">{genre_str}</div>
                <div style="display:flex;align-items:center;gap:4px;font-size:12px;">
                    <span style="font-size:13px;">⭐</span>
                    <span style="font-weight:600;color:#1a1a1a;">{rating_str}</span>
                    <span style="color:#aaa;font-size:11px;">/ 10</span>
                </div>
            </div>
        </div>

        <!-- Details grid -->
        <div style="border-top:0.5px solid #eee;padding:10px 12px;
                    display:grid;grid-template-columns:auto 1fr;
                    gap:4px 10px;font-size:11.5px;line-height:1.5;">
            <span style="color:#888;white-space:nowrap;">Location</span>
            <span style="color:#333;">{location_str}</span>

            <span style="color:#888;white-space:nowrap;">Neighborhood</span>
            <span style="color:#333;">{neighborhood_str}</span>

            <span style="color:#888;white-space:nowrap;">Director</span>
            <span style="color:#333;">{director_str}</span>

            <span style="color:#888;white-space:nowrap;">Box office</span>
            <span style="color:#333;">{boxing_str}</span>

            <span style="color:#888;white-space:nowrap;">Sentiment</span>
            <span style="display:flex;align-items:center;gap:5px;">
                <span style="display:inline-block;width:6px;height:6px;
                             border-radius:50%;background:{sent_color};
                             flex-shrink:0;"></span>
                <span style="color:#333;">{sentiment_raw}</span>
            </span>
        </div>
    </div>
    """
    return folium.Popup(html, max_width=260)

### Version 1

In [ ]:
m1 = folium.Map(
    location=[37.7749, -122.4194],
    zoom_start=12,
    tiles="CartoDB positron",
    control_scale=True
)

# Add marker cluster
cluster = MarkerCluster(
    options={
        "maxClusterRadius":    50,
        "disableClusteringAtZoom": 15,
    }
).add_to(m1)

# Add one marker per row
for _, row in df_map.iterrows():
    color = get_color(row["Primary_Genre"])

    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=7,
        color="white",
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=build_popup(row),
        tooltip=folium.Tooltip(
            f"<b>{row['Title']}</b> ({int(row['Year']) if pd.notna(row.get('Year')) else 'N/A'})"
            f"<br>{row.get('Locations', '')}",
            sticky=False
        ),
    ).add_to(cluster)

# ── Legend ────────────────────────────────────────────────────
legend_html = """
<div style="
    position: fixed;
    bottom: 30px; left: 30px;
    z-index: 1000;
    background: white;
    border: 0.5px solid #ccc;
    border-radius: 8px;
    padding: 12px 16px;
    font-family: sans-serif;
    font-size: 12px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.12);
    max-width: 160px;
">
    <div style="font-weight:600;margin-bottom:8px;font-size:13px;">Genre</div>
"""
for genre, color in GENRE_COLORS.items():
    if genre == "Other":
        continue
    legend_html += f"""
    <div style="display:flex;align-items:center;gap:7px;margin-bottom:5px;">
        <div style="width:11px;height:11px;border-radius:50%;
                    background:{color};flex-shrink:0;"></div>
        <span style="color:#333;">{genre}</span>
    </div>"""
legend_html += "</div>"

m1.get_root().html.add_child(folium.Element(legend_html))

# ── Title banner ──────────────────────────────────────────────
title_html = """
<div style="
    position: fixed;
    top: 12px; left: 50%;
    transform: translateX(-50%);
    z-index: 1000;
    background: white;
    border: 0.5px solid #ccc;
    border-radius: 8px;
    padding: 8px 20px;
    font-family: sans-serif;
    font-size: 13px;
    font-weight: 600;
    color: #1a1a1a;
    box-shadow: 0 2px 8px rgba(0,0,0,0.10);
    pointer-events: none;
">
    San Francisco Filming Locations · Click a marker to explore
</div>
"""
m1.get_root().html.add_child(folium.Element(title_html))

m1.save("sf_map_clustered.html")
print("Saved sf_map_clustered.html")

Saved sf_map_clustered.html


### Version 2

In [11]:
def build_popup(row):
    # Show all genres in the popup badge row
    all_genres = row.get("All_Genres", [row.get("Primary_Genre", "N/A")])
    genre_badges = "".join(
        f'<span style="display:inline-flex;align-items:center;gap:3px;'
        f'background:{GENRE_COLORS.get(g, "#888")}22;border-radius:4px;'
        f'padding:2px 6px;font-size:10px;color:{GENRE_COLORS.get(g, "#888")};'
        f'font-weight:500;margin-right:3px;">{g}</span>'
        for g in all_genres
    )

    # ... rest of your existing build_popup fields ...

    html = f"""
    <div style="font-family:sans-serif;font-size:12px;width:240px;
                background:#fff;border-radius:10px;overflow:hidden;
                line-height:1.5;box-shadow:0 2px 12px rgba(0,0,0,0.12);">
        <div style="display:flex;align-items:stretch;">
            {poster_html}
            <div style="padding:10px 12px;display:flex;flex-direction:column;
                        justify-content:center;min-width:0;gap:4px;">
                <div style="font-size:14px;font-weight:600;color:#1a1a1a;
                            line-height:1.3;">{row['Title']}</div>
                <div style="font-size:11px;color:#888;">{year_str}</div>
                <div style="display:flex;flex-wrap:wrap;gap:2px;margin-top:1px;">
                    {genre_badges}
                </div>
                <div style="display:flex;align-items:center;gap:4px;font-size:12px;">
                    <span>⭐</span>
                    <span style="font-weight:600;color:#1a1a1a;">{rating_str}</span>
                    <span style="color:#aaa;font-size:11px;">/ 10</span>
                </div>
            </div>
        </div>
        <div style="border-top:0.5px solid #eee;padding:10px 12px;
                    display:grid;grid-template-columns:auto 1fr;
                    gap:4px 10px;font-size:11.5px;line-height:1.5;">
            <span style="color:#888;white-space:nowrap;">Location</span>
            <span style="color:#333;">{location_str}</span>
            <span style="color:#888;white-space:nowrap;">Neighborhood</span>
            <span style="color:#333;">{neighborhood_str}</span>
            <span style="color:#888;white-space:nowrap;">Director</span>
            <span style="color:#333;">{director_str}</span>
            <span style="color:#888;white-space:nowrap;">Box office</span>
            <span style="color:#333;">{boxing_str}</span>
            <span style="color:#888;white-space:nowrap;">Sentiment</span>
            <span style="display:flex;align-items:center;gap:5px;">
                <span style="display:inline-block;width:6px;height:6px;
                             border-radius:50%;background:{sent_color};flex-shrink:0;"></span>
                <span style="color:#333;">{sentiment_raw}</span>
            </span>
        </div>
    </div>
    """
    return folium.Popup(html, max_width=260)

m1.get_root().html.add_child(folium.Element(legend_html))
m1.get_root().html.add_child(folium.Element(filter_panel_html))

m1.save("sf_map_clustered2.html")
print("Saved sf_map_clustered2.html")

Saved sf_map_clustered2.html


### Density (heatmap) through decades

In [ ]:
import folium
from folium.plugins import HeatMapWithTime
import pandas as pd
import json

In [ ]:
import folium
from folium.plugins import HeatMapWithTime
import pandas as pd

# ── Prep ──────────────────────────────────────────────────────
df_heat = df_map.dropna(subset=["Latitude", "Longitude", "Year"]).copy()
df_heat["Year"] = df_heat["Year"].astype(int)
df_heat["Decade"] = (df_heat["Year"] // 10 * 10)

decades = sorted(df_heat["Decade"].unique())

heat_data_by_decade = []
time_labels = []

for decade in decades:
    cumulative = df_heat[df_heat["Decade"] <= decade]
    heat_data_by_decade.append(
        cumulative[["Latitude", "Longitude"]].values.tolist()
    )
    time_labels.append(f"{decade}s")

# ── Map ───────────────────────────────────────────────────────
m2a = folium.Map(
    location=[37.7749, -122.4194],
    zoom_start=12,
    tiles="CartoDB dark_matter",
)

HeatMapWithTime(
    heat_data_by_decade,
    index=time_labels,
    radius=20,
    blur=0.85,
    min_opacity=0.05,
    max_opacity=0.75,
    scale_radius=False,
    use_local_extrema=False,
    auto_play=True,
    display_index=True,
    index_steps=1,
    min_speed=0.1,
    max_speed=4,
    speed_step=0.5,
    position="bottomleft",
    gradient={
        "0.0": "#0d1b2a",   # near-black — lowest density, almost invisible
        "0.2": "#1a3a5c",   # deep navy
        "0.45": "#1d6a8a",  # teal-blue mid
        "0.65": "#c47a2b",  # warm amber — hotspots start to glow
        "0.85": "#b84c2b",  # burnt coral
        "1.0": "#e8d5b0",   # warm off-white — only the absolute peaks
    },
).add_to(m2a)

# ── Dark theme slider styles ───────────────────────────────────
slider_style = """
<style>
  .timecontrol-slider          { background: rgba(20,20,20,0.85) !important; }
  .timecontrol-date            { color: #fff !important; font-family: sans-serif; font-size: 13px; }
  .leaflet-control-timecontrol { border: 0.5px solid #555 !important; border-radius: 8px !important; }
  .timecontrol-play            { background: rgba(20,20,20,0.85) !important; color: #fff !important; }
  .timecontrol-speed span      { color: #aaa !important; }
</style>
"""
m2a.get_root().html.add_child(folium.Element(slider_style))

# ── Title ─────────────────────────────────────────────────────
title_html = """
<div style="
    position:fixed; top:12px; left:50%; transform:translateX(-50%);
    z-index:1000; background:rgba(20,20,20,0.85);
    border:0.5px solid #555; border-radius:8px;
    padding:8px 20px; font-family:sans-serif;
    font-size:13px; font-weight:600; color:white;
    pointer-events:none; white-space:nowrap;">
  Filming Location Density — San Francisco
  <span style="font-weight:400; color:#aaa; margin-left:8px;">cumulative by decade</span>
</div>
"""
m2a.get_root().html.add_child(folium.Element(title_html))

# ── Ghost decade label (bottom-right, updates with slider) ─────
decade_overlay = """
<div id="decade-label" style="
    position:fixed; bottom:80px; right:20px; z-index:1000;
    font-family:'Helvetica Neue',sans-serif;
    font-size:48px; font-weight:700;
    color:rgba(255,255,255,0.12);
    pointer-events:none; letter-spacing:-1px;">
</div>
<script>
(function() {
  function updateLabel() {
    var dateEl = document.querySelector('.timecontrol-date');
    if (dateEl) {
      document.getElementById('decade-label').textContent = dateEl.textContent.trim();
    }
    setTimeout(updateLabel, 300);
  }
  window.addEventListener('load', function() { setTimeout(updateLabel, 1000); });
})();
</script>
"""
m2a.get_root().html.add_child(folium.Element(decade_overlay))

m2a.save("sf_map_heatmap_animated.html")
print("Saved sf_map_heatmap_animated.html")

Saved sf_map_heatmap_animated.html


### Neighborhood division of the map

In [9]:
import folium
import pandas as pd
import json
import math

In [12]:
# ── Load neighborhood GeoJSON ─────────────────────────────────
with open("SF_Find_Neighborhoods_20260428.geojson", "r") as f:
    neighborhoods_geojson = json.load(f)

# Inspect what property holds the neighborhood name — adjust if needed
# Common values: "nhood", "neighborhood", "name"
NAME_FIELD = "name"


In [13]:

# ── Count films per neighborhood ──────────────────────────────
# Use the "Analysis neighborhood" column which was already geocoded
neighborhood_counts = (
    df_map
    .dropna(subset=["Analysis neighborhood"])
    .groupby("Analysis neighborhood")
    .size()
    .reset_index(name="film_count")
    .rename(columns={"Analysis neighborhood": "neighborhood"})
)

count_map = dict(zip(
    neighborhood_counts["neighborhood"].str.strip().str.lower(),
    neighborhood_counts["film_count"]
))

# Attach count to each GeoJSON feature for choropleth
for feature in neighborhoods_geojson["features"]:
    raw_name = feature["properties"].get(NAME_FIELD, "")
    feature["properties"]["film_count"] = count_map.get(
        str(raw_name).strip().lower(), 0
    )

# ── Compute neighborhood centroids for circle markers ─────────
def geojson_centroid(geometry):
    """Rough centroid from bounding box — good enough for labels."""
    coords = []
    def collect(c):
        if isinstance(c[0], list):
            for sub in c: collect(sub)
        else:
            coords.append(c)
    collect(geometry["coordinates"])
    lngs = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return (
        (min(lats) + max(lats)) / 2,
        (min(lngs) + max(lngs)) / 2,
    )

centroids = []
for feature in neighborhoods_geojson["features"]:
    name   = feature["properties"].get(NAME_FIELD, "")
    count  = feature["properties"]["film_count"]
    lat, lng = geojson_centroid(feature["geometry"])
    centroids.append({"name": name, "count": count, "lat": lat, "lng": lng})

max_count = max(c["count"] for c in centroids) if centroids else 1

# ── Map ───────────────────────────────────────────────────────
m2b = folium.Map(
    location=[37.7749, -122.4194],
    zoom_start=12,
    tiles="CartoDB dark_matter",
)

# ── Neighborhood fill (choropleth-style via GeoJson) ──────────
def film_count_style(feature):
    count    = feature["properties"].get("film_count", 0)
    fraction = count / max_count if max_count > 0 else 0

    # Interpolate: dark blue (sparse) → amber → coral (dense)
    if fraction < 0.5:
        t = fraction / 0.5
        r = int(24  + t * (186 - 24))
        g = int(95  + t * (117 - 95))
        b = int(165 + t * (23  - 165))
    else:
        t = (fraction - 0.5) / 0.5
        r = int(186 + t * (153 - 186))
        g = int(117 + t * (53  - 117))
        b = int(23  + t * (29  - 23))

    fill_color = f"#{r:02x}{g:02x}{b:02x}"
    opacity    = 0.15 + fraction * 0.35   # 0.15 → 0.50

    return {
        "fillColor":   fill_color,
        "fillOpacity": opacity,
        "color":       "#aaaaaa",         # border colour
        "weight":      0.8,
        "dashArray":   "3 3",
    }

folium.GeoJson(
    neighborhoods_geojson,
    name="Neighborhoods",
    style_function=film_count_style,
    tooltip=folium.GeoJsonTooltip(
        fields=[NAME_FIELD, "film_count"],
        aliases=["Neighborhood", "Films recorded"],
        localize=True,
        sticky=False,
        style=(
            "background-color:rgba(20,20,20,0.85);"
            "color:white;"
            "font-family:sans-serif;"
            "font-size:12px;"
            "border:0.5px solid #555;"
            "border-radius:6px;"
            "padding:6px 10px;"
        ),
    ),
).add_to(m2b)

# ── Circle markers at centroids ───────────────────────────────
for c in centroids:
    if c["count"] == 0:
        continue

    fraction  = c["count"] / max_count
    # Radius: 10px minimum, 44px maximum
    radius    = 10 + fraction * 34
    # Opacity: 0.55 minimum, 0.92 maximum
    opacity   = 0.55 + fraction * 0.37

    folium.CircleMarker(
        location=[c["lat"], c["lng"]],
        radius=radius,
        color="white",
        weight=0.8,
        fill=True,
        fill_color="#BA7517",
        fill_opacity=opacity,
        tooltip=folium.Tooltip(
            f"<b style='font-family:sans-serif;color:white;'>{c['name']}</b>"
            f"<br><span style='color:#ccc;font-family:sans-serif;font-size:11px;'>"
            f"{c['count']} filming location{'s' if c['count'] != 1 else ''}</span>",
            sticky=False,
        ),
        popup=folium.Popup(
            f"""<div style='font-family:sans-serif;font-size:12px;
                            min-width:140px;text-align:center;'>
                  <div style='font-size:15px;font-weight:600;
                              color:#1a1a1a;margin-bottom:4px;'>{c['name']}</div>
                  <div style='font-size:28px;font-weight:700;
                              color:#BA7517;line-height:1;'>{c['count']}</div>
                  <div style='font-size:11px;color:#888;margin-top:2px;'>
                    filming location{'s' if c['count'] != 1 else ''}
                  </div>
                </div>""",
            max_width=180,
        ),
    ).add_to(m2b)

    # Count label directly on the circle
    folium.Marker(
        location=[c["lat"], c["lng"]],
        icon=folium.DivIcon(
            html=f"""<div style="
                font-family:'Helvetica Neue',sans-serif;
                font-size:{'11' if c['count'] < 100 else '10'}px;
                font-weight:700; color:white;
                text-align:center; line-height:1;
                text-shadow:0 1px 3px rgba(0,0,0,0.8);
                pointer-events:none;
                transform:translate(-50%,-50%);
                width:60px; margin-left:-30px; margin-top:-7px;
            ">{c['count']}</div>""",
            icon_size=(60, 20),
            icon_anchor=(30, 10),
        ),
    ).add_to(m2b)

# ── Colour scale legend ───────────────────────────────────────
legend_html = f"""
<div style="
    position:fixed; bottom:30px; left:16px; z-index:1000;
    background:rgba(20,20,20,0.85); border:0.5px solid #555;
    border-radius:10px; padding:12px 16px;
    font-family:'Helvetica Neue',sans-serif; font-size:11.5px;">
  <div style="color:#ccc;font-weight:600;margin-bottom:8px;
              font-size:12px;letter-spacing:0.03em;">Films per neighborhood</div>
  <div style="display:flex;align-items:center;gap:6px;">
    <span style="color:#888;font-size:10px;">Fewer</span>
    <div style="
        width:110px;height:8px;border-radius:4px;
        background:linear-gradient(to right,#185FA5,#BA7517,#993C1D);
    "></div>
    <span style="color:#888;font-size:10px;">More</span>
  </div>
  <div style="color:#888;font-size:10px;margin-top:8px;">
    Max: <span style="color:#ccc;font-weight:600;">{max_count}</span> locations
  </div>
</div>
"""
m2b.get_root().html.add_child(folium.Element(legend_html))

# ── Title ─────────────────────────────────────────────────────
title_html2b = """
<div style="
    position:fixed; top:12px; left:50%; transform:translateX(-50%);
    z-index:1000; background:rgba(20,20,20,0.85);
    border:0.5px solid #555; border-radius:8px;
    padding:8px 20px; font-family:sans-serif;
    font-size:13px; font-weight:600; color:white;
    pointer-events:none; white-space:nowrap;">
  Filming Locations by Neighborhood — San Francisco
  <span style="font-weight:400; color:#aaa; margin-left:8px;">hover or click a circle</span>
</div>
"""
m2b.get_root().html.add_child(folium.Element(title_html2b))

m2b.save("sf_map_neighborhoods.html")
print("Saved sf_map_neighborhoods.html")

Saved sf_map_neighborhoods.html


In [14]:
# See what property keys are available
print("Feature count:", len(neighborhoods_geojson["features"]))
print("Sample properties:", neighborhoods_geojson["features"][0]["properties"])
print("Geometry type:", neighborhoods_geojson["features"][0]["geometry"]["type"])

Feature count: 117
Sample properties: {':id': 'row-9c9s-nrvx_d2st', ':version': 'rv-ik5x~5ee5~qn5j', ':created_at': '2023-10-17T18:16:15.759Z', ':updated_at': '2023-10-17T18:16:15.759Z', 'link': 'http://en.wikipedia.org/wiki/Sea_Cliff,_San_Francisco,_California', 'name': 'Seacliff', 'film_count': 2}
Geometry type: MultiPolygon


In [15]:
# ── 2. See all neighborhood names in the GeoJSON ─────────────
# Replace "nhood" below with whatever key appeared in the output above
NAME_FIELD = "name"

geojson_names = sorted([
    f["properties"].get(NAME_FIELD, "MISSING")
    for f in neighborhoods_geojson["features"]
])
print("GeoJSON neighborhood names:")
for n in geojson_names:
    print(" ", repr(n))

GeoJSON neighborhood names:
  'Alamo Square'
  'Anza Vista'
  'Apparel City'
  'Aquatic Park / Ft. Mason'
  'Ashbury Heights'
  'Balboa Terrace'
  'Bayview'
  'Bernal Heights'
  'Bret Harte'
  'Buena Vista'
  'Candlestick Point SRA'
  'Castro'
  'Cathedral Hill'
  'Cayuga'
  'Central Waterfront'
  'Chinatown'
  'Civic Center'
  'Clarendon Heights'
  'Cole Valley'
  'Corona Heights'
  'Cow Hollow'
  'Crocker Amazon'
  'Diamond Heights'
  'Dogpatch'
  'Dolores Heights'
  'Downtown / Union Square'
  'Duboce Triangle'
  'Eureka Valley'
  'Excelsior'
  'Fairmount'
  'Financial District'
  'Fishermans Wharf'
  'Forest Hill'
  'Forest Knolls'
  'Glen Park'
  'Golden Gate Heights'
  'Golden Gate Park'
  'Haight Ashbury'
  'Hayes Valley'
  'Holly Park'
  'Hunters Point'
  'India Basin'
  'Ingleside'
  'Ingleside Terraces'
  'Inner Richmond'
  'Inner Sunset'
  'Japantown'
  'Laguna Honda'
  'Lake Street'
  'Lakeshore'
  'Laurel Heights / Jordan Park'
  'Lincoln Park / Ft. Miley'
  'Little Hollyw

In [16]:
# ── 3. See all neighborhood names in your dataframe ───────────
df_names = sorted(df_map["Analysis neighborhood"].dropna().unique())
print("\nDataFrame neighborhood names:")
for n in df_names:
    print(" ", repr(n))


DataFrame neighborhood names:
  'Bayview Hunters Point'
  'Bernal Heights'
  'Castro/Upper Market'
  'Chinatown'
  'Excelsior'
  'Financial District/South Beach'
  'Glen Park'
  'Golden Gate Park'
  'Haight Ashbury'
  'Hayes Valley'
  'Inner Richmond'
  'Inner Sunset'
  'Japantown'
  'Lakeshore'
  'Lincoln Park'
  'Lone Mountain/USF'
  'Marina'
  'McLaren Park'
  'Mission'
  'Mission Bay'
  'Nob Hill'
  'Noe Valley'
  'North Beach'
  'Oceanview/Merced/Ingleside'
  'Outer Mission'
  'Outer Richmond'
  'Pacific Heights'
  'Portola'
  'Potrero Hill'
  'Presidio'
  'Presidio Heights'
  'Russian Hill'
  'Seacliff'
  'South of Market'
  'Sunset/Parkside'
  'Tenderloin'
  'Treasure Island'
  'Twin Peaks'
  'West of Twin Peaks'
  'Western Addition'


In [17]:
# ── 4. Check which ones match and which don't ─────────────────
geojson_set = {n.strip().lower() for n in geojson_names}
df_set      = {n.strip().lower() for n in df_names}

print("\nIn DataFrame but NOT in GeoJSON (will show 0 count):")
for n in sorted(df_set - geojson_set):
    print(" ", repr(n))

print("\nIn GeoJSON but NOT in DataFrame (will be empty on map):")
for n in sorted(geojson_set - df_set):
    print(" ", repr(n))

print("\nMatched successfully:", len(df_set & geojson_set), "neighborhoods")


In DataFrame but NOT in GeoJSON (will show 0 count):
  'bayview hunters point'
  'castro/upper market'
  'financial district/south beach'
  'lincoln park'
  'lone mountain/usf'
  'oceanview/merced/ingleside'
  'presidio'
  'sunset/parkside'
  'twin peaks'
  'west of twin peaks'

In GeoJSON but NOT in DataFrame (will be empty on map):
  'alamo square'
  'anza vista'
  'apparel city'
  'aquatic park / ft. mason'
  'ashbury heights'
  'balboa terrace'
  'bayview'
  'bret harte'
  'buena vista'
  'candlestick point sra'
  'castro'
  'cathedral hill'
  'cayuga'
  'central waterfront'
  'civic center'
  'clarendon heights'
  'cole valley'
  'corona heights'
  'cow hollow'
  'crocker amazon'
  'diamond heights'
  'dogpatch'
  'dolores heights'
  'downtown / union square'
  'duboce triangle'
  'eureka valley'
  'fairmount'
  'financial district'
  'fishermans wharf'
  'forest hill'
  'forest knolls'
  'golden gate heights'
  'holly park'
  'hunters point'
  'india basin'
  'ingleside'
  'ingl

### More correct approach for the geospatial neighborhoods

In [20]:
import folium
import pandas as pd
import json
import urllib.request
from shapely.geometry import Point, shape

In [21]:
# ── Load neighborhood GeoJSON ─────────────────────────────────
with open("SF_Find_Neighborhoods_20260428.geojson", "r") as f:
    neighborhoods_geojson = json.load(f)

# Inspect what property holds the neighborhood name — adjust if needed
# Common values: "nhood", "neighborhood", "name"
NAME_FIELD = "name"


In [35]:
import folium
import json
from shapely.geometry import Point, shape

# ── Load and clean GeoJSON ────────────────────────────────────
with open("SF_Find_Neighborhoods_20260428.geojson", "r") as f:
    raw_geojson = json.load(f)

NAME_FIELD = "name"

clean_features = []
for feature in raw_geojson["features"]:
    clean_props = {
        k: v for k, v in feature["properties"].items()
        if not k.startswith(":")
    }
    clean_features.append({
        "type":       "Feature",
        "geometry":   feature["geometry"],
        "properties": clean_props,
    })

neighborhoods_geojson = {
    "type":     "FeatureCollection",
    "features": clean_features,
}

# ── Build shapely geometries ──────────────────────────────────
geo_shapes = []
for feature in neighborhoods_geojson["features"]:
    geo_shapes.append({
        "name":     feature["properties"][NAME_FIELD].strip(),
        "geometry": shape(feature["geometry"]),
    })

# ── Point-in-polygon assignment ───────────────────────────────
def find_neighborhood(lat, lng):
    pt = Point(lng, lat)
    for gs in geo_shapes:
        if gs["geometry"].contains(pt):
            return gs["name"]
    min_dist, nearest = float("inf"), None
    for gs in geo_shapes:
        dist = gs["geometry"].centroid.distance(pt)
        if dist < min_dist:
            min_dist = dist
            nearest  = gs["name"]
    return nearest

df_clean = df_map.dropna(subset=["Latitude", "Longitude"]).copy()
df_clean["geo_neighborhood"] = df_clean.apply(
    lambda row: find_neighborhood(row["Latitude"], row["Longitude"]),
    axis=1
)

# ── Build counts ──────────────────────────────────────────────
count_map = df_clean["geo_neighborhood"].value_counts().to_dict()
max_count = max(count_map.values()) if count_map else 1

# ── Attach counts to GeoJSON ──────────────────────────────────
for feature in neighborhoods_geojson["features"]:
    name = feature["properties"][NAME_FIELD].strip()
    feature["properties"]["film_count"] = int(count_map.get(name, 0))

# ── Style function ────────────────────────────────────────────
def film_count_style(feature):
    count    = feature["properties"].get("film_count", 0)
    fraction = count / max_count

    if fraction == 0:
        return {
            "fillColor":   "#0a0f1a",
            "fillOpacity": 0.75,
            "color":       "#2a2a3a",
            "weight":      0.5,
        }

    if fraction < 0.5:
        t = fraction / 0.5
        r = int(13  + t * (186 - 13))
        g = int(27  + t * (117 - 27))
        b = int(42  + t * (23  - 42))
    else:
        t = (fraction - 0.5) / 0.5
        r = int(186 + t * (232 - 186))
        g = int(117 + t * (213 - 117))
        b = int(23  + t * (176 - 23))

    return {
        "fillColor":   f"#{r:02x}{g:02x}{b:02x}",
        "fillOpacity": 0.55 + fraction * 0.40,
        "color":       "#aaaaaa",
        "weight":      0.6,
    }

# ── Build label data for JS ───────────────────────────────────
# Pass centroid + count to JavaScript — label positions are then
# recomputed in pixel space on every zoom/move, so they stay centred
label_data = []
for gs in geo_shapes:
    name     = gs["name"]
    count    = int(count_map.get(name, 0))
    centroid = gs["geometry"].centroid
    fraction = count / max_count
    # Light text on dark-blue/teal polygons, dark text only on the
    # bright warm off-white peak (fraction > 0.92)
    text_color = "#1a1a1a" if fraction > 0.92 else "#ffffff"
    label_data.append({
        "lat":      centroid.y,
        "lng":      centroid.x,
        "label":    str(count) if count > 0 else "—",
        "color":    text_color,
        "opacity":  0.25 if count == 0 else round(0.65 + fraction * 0.35, 2),
        "fontSize": 9 if count >= 100 else 11,
    })

# ── Map ───────────────────────────────────────────────────────
m2b = folium.Map(
    location=[37.7749, -122.4194],
    zoom_start=12,
    tiles="CartoDB dark_matter",
)

folium.GeoJson(
    neighborhoods_geojson,
    name="Neighborhoods",
    style_function=film_count_style,
    tooltip=folium.GeoJsonTooltip(
        fields=[NAME_FIELD, "film_count"],
        aliases=["Neighborhood", "Films recorded"],
        localize=True,
        sticky=False,
        style=(
            "background-color:rgba(20,20,20,0.90);"
            "color:white;"
            "font-family:'Helvetica Neue',sans-serif;"
            "font-size:12px;"
            "border:0.5px solid #555;"
            "border-radius:6px;"
            "padding:6px 10px;"
        ),
    ),
).add_to(m2b)

# ── JS label layer ────────────────────────────────────────────
# Uses a single absolutely-positioned <canvas>-like div overlay.
# On every 'moveend' and 'zoomend' event, Leaflet converts each
# centroid (lat/lng) to pixel coordinates and repositions the label
# div to match — so labels always sit in the visual centre of their
# polygon at every zoom level, with no drift.
import json as _json

labels_js = f"""
<script>
(function() {{
  var LABELS = {_json.dumps(label_data)};

  window.addEventListener('load', function() {{

    // Find the Folium map object
    var leafletMap = null;
    for (var key in window) {{
      if (key.startsWith('map_') && window[key] && window[key].eachLayer) {{
        leafletMap = window[key];
        break;
      }}
    }}
    if (!leafletMap) return;

    // Create a single overlay pane for all labels
    var pane = leafletMap.getPane('overlayPane');
    var container = document.createElement('div');
    container.id  = 'neighborhood-labels';
    container.style.cssText = [
      'position:absolute',
      'top:0', 'left:0',
      'width:100%', 'height:100%',
      'pointer-events:none',
      'z-index:400',
    ].join(';');
    pane.appendChild(container);

    // Build one div per label
    var divs = LABELS.map(function(d) {{
      var el       = document.createElement('div');
      el.textContent = d.label;
      el.style.cssText = [
        'position:absolute',
        'font-family:Helvetica Neue,sans-serif',
        'font-size:' + d.font_size + 'px',
        'font-weight:700',
        'color:' + d.color,
        'opacity:' + d.opacity,
        'text-shadow:0 1px 3px rgba(0,0,0,0.7)',
        'white-space:nowrap',
        'transform:translate(-50%,-50%)',  // centre on the coordinate point
        'will-change:transform',
      ].join(';');
      container.appendChild(el);
      return el;
    }});

    // Reposition every label in pixel space
    function updatePositions() {{
      // The overlay pane shifts with map.getPixelOrigin() — we must
      // account for this offset so labels track the map tiles exactly
      var origin = leafletMap.getPixelOrigin();
      LABELS.forEach(function(d, i) {{
        var pt = leafletMap.project(L.latLng(d.lat, d.lng));
        var x  = pt.x - origin.x;
        var y  = pt.y - origin.y;
        divs[i].style.left = x + 'px';
        divs[i].style.top  = y + 'px';
      }});
    }}

    // Run on load and whenever the map moves or zooms
    updatePositions();
    leafletMap.on('moveend zoomend viewreset move', updatePositions);
  }});
}})();
</script>
"""

m2b.get_root().html.add_child(folium.Element(labels_js))

# ── Legend ────────────────────────────────────────────────────
m2b.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;bottom:30px;left:16px;z-index:1000;
    background:rgba(20,20,20,0.88);border:0.5px solid #555;
    border-radius:10px;padding:12px 16px;
    font-family:'Helvetica Neue',sans-serif;font-size:11.5px;">
  <div style="color:#ccc;font-weight:600;margin-bottom:8px;
              font-size:12px;letter-spacing:0.03em;">Filming locations</div>
  <div style="display:flex;align-items:center;gap:8px;margin-bottom:8px;">
    <span style="color:#777;font-size:10px;">0</span>
    <div style="width:110px;height:7px;border-radius:3px;
        background:linear-gradient(to right,#0d1b2a,#1a3a5c,#1d6a8a,#c47a2b,#b84c2b,#e8d5b0);"></div>
    <span style="color:#777;font-size:10px;">{max_count}</span>
  </div>
  <div style="color:#666;font-size:10px;">
    per neighborhood · hover for details
  </div>
</div>
"""))

# ── Title ─────────────────────────────────────────────────────
m2b.get_root().html.add_child(folium.Element("""
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
    z-index:1000;background:rgba(20,20,20,0.85);
    border:0.5px solid #555;border-radius:8px;
    padding:8px 20px;font-family:sans-serif;
    font-size:13px;font-weight:600;color:white;
    pointer-events:none;white-space:nowrap;">
  Filming Locations by Neighborhood — San Francisco
  <span style="font-weight:400;color:#aaa;margin-left:8px;">hover a neighborhood for details</span>
</div>
"""))

m2b.save("sf_map_neighborhoods.html")
print("Saved sf_map_neighborhoods.html")

Saved sf_map_neighborhoods.html


### Combination of graphs

In [ ]:
from shapely.geometry import Point, shape
import json

# ════════════════════════════════════════════════════════════════
# 1. BUILD MAP A DATA (heatmap frames)
# ════════════════════════════════════════════════════════════════
df_heat = df_map.dropna(subset=["Latitude", "Longitude", "Year"]).copy()
df_heat["Year"]   = df_heat["Year"].astype(int)
df_heat["Decade"] = (df_heat["Year"] // 10 * 10)

decades = sorted(df_heat["Decade"].unique())
heat_data_by_decade = []
time_labels = []

for decade in decades:
    cumulative = df_heat[df_heat["Decade"] <= decade]
    heat_data_by_decade.append(
        cumulative[["Latitude", "Longitude"]].values.tolist()
    )
    time_labels.append(f"{decade}s")

# ════════════════════════════════════════════════════════════════
# 2. BUILD MAP B DATA (neighborhood choropleth)
# ════════════════════════════════════════════════════════════════
with open("SF_Find_Neighborhoods_20260428.geojson", "r") as f:
    raw_geojson = json.load(f)

NAME_FIELD = "name"

clean_features = []
for feature in raw_geojson["features"]:
    clean_props = {k: v for k, v in feature["properties"].items()
                   if not k.startswith(":")}
    clean_features.append({
        "type":       "Feature",
        "geometry":   feature["geometry"],
        "properties": clean_props,
    })

neighborhoods_geojson = {"type": "FeatureCollection", "features": clean_features}

geo_shapes = [
    {"name": f["properties"][NAME_FIELD].strip(), "geometry": shape(f["geometry"])}
    for f in neighborhoods_geojson["features"]
]

def find_neighborhood(lat, lng):
    pt = Point(lng, lat)
    for gs in geo_shapes:
        if gs["geometry"].contains(pt):
            return gs["name"]
    return min(geo_shapes, key=lambda gs: gs["geometry"].centroid.distance(pt))["name"]

df_clean = df_map.dropna(subset=["Latitude", "Longitude"]).copy()
df_clean["geo_neighborhood"] = df_clean.apply(
    lambda row: find_neighborhood(row["Latitude"], row["Longitude"]), axis=1
)

count_map = df_clean["geo_neighborhood"].value_counts().to_dict()
max_count = max(count_map.values()) if count_map else 1

for feature in neighborhoods_geojson["features"]:
    name = feature["properties"][NAME_FIELD].strip()
    feature["properties"]["film_count"] = int(count_map.get(name, 0))

def get_fill(count):
    fraction = count / max_count
    if fraction == 0:
        return "#0d1b2a", 0.20   # dark navy, nearly transparent

    # Interpolate across the same 5 segments as the heatmap gradient
    stops = [
        (0.00, (13,  27,  42)),   # #0d1b2a
        (0.20, (26,  58,  92)),   # #1a3a5c
        (0.45, (29, 106, 138)),   # #1d6a8a
        (0.65, (196, 122, 43)),   # #c47a2b
        (0.85, (184,  76, 43)),   # #b84c2b
        (1.00, (232, 213, 176)),  # #e8d5b0
    ]

    # Find which segment fraction falls in and interpolate
    for i in range(len(stops) - 1):
        t0, c0 = stops[i]
        t1, c1 = stops[i + 1]
        if fraction <= t1:
            t = (fraction - t0) / (t1 - t0)
            r = int(c0[0] + t * (c1[0] - c0[0]))
            g = int(c0[1] + t * (c1[1] - c0[1]))
            b = int(c0[2] + t * (c1[2] - c0[2]))
            fill_opacity = 0.30 + fraction * 0.65  # 0.30 → 0.95
            return f"#{r:02x}{g:02x}{b:02x}", round(fill_opacity, 3)

    return "#e8d5b0", 0.95

label_data = []
for gs in geo_shapes:
    name     = gs["name"]
    count    = int(count_map.get(name, 0))
    centroid = gs["geometry"].centroid
    fraction = count / max_count
    label_data.append({
        "lat":      centroid.y,
        "lng":      centroid.x,
        "label":    str(count) if count > 0 else "—",
        "color":    "#1a1a1a" if fraction > 0.55 else "#ffffff",
        "opacity":  0.35 if count == 0 else round(0.70 + fraction * 0.30, 2),
        "fontSize": 9 if count >= 100 else 11,
    })

feature_styles = {}
for feature in neighborhoods_geojson["features"]:
    name  = feature["properties"][NAME_FIELD].strip()
    count = feature["properties"]["film_count"]
    fill, fill_opacity = get_fill(count)
    feature_styles[name] = {
        "fillColor":   fill,
        "fillOpacity": fill_opacity,
        "color":       "#aaaaaa" if count > 0 else "#2a2a3a",
        "weight":      0.6 if count > 0 else 0.5,
    }

# ════════════════════════════════════════════════════════════════
# 3. Serialise Python data to JS literals
# ════════════════════════════════════════════════════════════════
heat_frames_js    = json.dumps(heat_data_by_decade)
time_labels_js    = json.dumps(time_labels)
geojson_js        = json.dumps(neighborhoods_geojson)
feature_styles_js = json.dumps(feature_styles)
label_data_js     = json.dumps(label_data)
max_count_js      = str(max_count)
name_field_js     = json.dumps(NAME_FIELD)

# ════════════════════════════════════════════════════════════════
# 4. ASSEMBLE HTML
#    No f-string for the JS block — use .format() with named keys
#    only for the Python-computed values, keeping all JS braces safe
# ════════════════════════════════════════════════════════════════

# CSS and HTML structure use an f-string (no JS braces here)
html_head = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>SF Filming Locations</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css">
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet.heat/0.2.0/leaflet-heat.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ background:#0d0d0d; overflow:hidden; font-family:'Helvetica Neue',sans-serif; }}

  #map-a, #map-b {{
    position:fixed; inset:0; width:100vw; height:100vh;
    transition: opacity 1.4s cubic-bezier(0.4, 0, 0.2, 1);
  }}
  #map-b {{ opacity:0; pointer-events:none; }}

  #decade-ghost {{
    position:fixed; bottom:80px; right:32px;
    font-size:72px; font-weight:700;
    color:rgba(255,255,255,0.10);
    letter-spacing:-2px; pointer-events:none;
    z-index:800; transition:opacity 0.5s; user-select:none;
  }}

  .map-title {{
    position:fixed; top:16px; left:50%; transform:translateX(-50%);
    z-index:900; background:rgba(20,20,20,0.85);
    border:0.5px solid #555; border-radius:8px;
    padding:8px 22px; font-size:13px; font-weight:600;
    color:white; pointer-events:none; white-space:nowrap;
    transition:opacity 0.6s;
  }}

  #scroll-hint {{
    position:fixed; bottom:28px; left:50%; transform:translateX(-50%);
    z-index:900; color:rgba(255,255,255,0.45); font-size:12px;
    letter-spacing:0.08em; pointer-events:none; transition:opacity 0.6s;
    display:flex; flex-direction:column; align-items:center; gap:5px;
  }}
  #scroll-hint .arrow {{
    width:0; height:0;
    border-left:5px solid transparent;
    border-right:5px solid transparent;
    border-top:6px solid rgba(255,255,255,0.45);
    animation:bounce 1.4s infinite;
  }}
  @keyframes bounce {{
    0%,100% {{ transform:translateY(0); }}
    50%     {{ transform:translateY(4px); }}
  }}

  #legend-b {{
    position:fixed; bottom:30px; left:16px; z-index:900;
    background:rgba(20,20,20,0.88); border:0.5px solid #555;
    border-radius:10px; padding:12px 16px; font-size:11.5px;
    opacity:0; pointer-events:none;
    transition:opacity 1.4s cubic-bezier(0.4, 0, 0.2, 1);
  }}

  .dark-tooltip {{
    background:rgba(20,20,20,0.90) !important;
    color:white !important;
    font-family:'Helvetica Neue',sans-serif !important;
    font-size:12px !important;
    border:0.5px solid #555 !important;
    border-radius:6px !important;
    padding:6px 10px !important;
  }}
  .leaflet-tooltip.dark-tooltip::before {{ border-top-color:#555 !important; }}
</style>
</head>
<body>

<div id="map-a"></div>
<div id="map-b"></div>
<div id="decade-ghost"></div>

<div class="map-title" id="title-a">
  Filming Location Density — San Francisco
  <span style="font-weight:400;color:#aaa;margin-left:8px;">scroll to advance</span>
</div>
<div class="map-title" id="title-b" style="opacity:0;">
  Filming Locations by Neighborhood — San Francisco
  <span style="font-weight:400;color:#aaa;margin-left:8px;">hover a neighborhood</span>
</div>

<div id="scroll-hint">
  <span>scroll</span>
  <div class="arrow"></div>
</div>

<div id="legend-b">
  <div style="color:#ccc;font-weight:600;margin-bottom:8px;font-size:12px;letter-spacing:0.03em;">Filming locations</div>
  <div style="display:flex;align-items:center;gap:8px;margin-bottom:6px;">
    <span style="color:#777;font-size:10px;">0</span>
    <div style="width:110px;height:7px;border-radius:3px;
                background:linear-gradient(to right,#0d1b2a,#1a3a5c,#1d6a8a,#c47a2b,#b84c2b,#e8d5b0);"></div>
    <span style="color:#777;font-size:10px;">{max_count}</span>
  </div>
  <div style="color:#666;font-size:10px;">per neighborhood · hover for details</div>
</div>
"""

# JS block: plain string (no f-string), Python values injected via .replace()
# This avoids ALL f-string brace-escaping issues in JS
html_js = """
<script>
// ── Data ──────────────────────────────────────────────────────
var HEAT_FRAMES    = __HEAT_FRAMES__;
var TIME_LABELS    = __TIME_LABELS__;
var GEOJSON        = __GEOJSON__;
var FEATURE_STYLES = __FEATURE_STYLES__;
var LABEL_DATA     = __LABEL_DATA__;
var NAME_FIELD     = __NAME_FIELD__;

// ── Shared config ─────────────────────────────────────────────
var MAP_CENTER = [37.7749, -122.4194];
var MAP_ZOOM   = 13;
var TILE_URL   = 'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png';
var TILE_OPTS  = { maxZoom: 19 };

// ── Map A ─────────────────────────────────────────────────────
var mapA = L.map('map-a', {
  center: MAP_CENTER, zoom: MAP_ZOOM,
  zoomControl: false, attributionControl: false,
  scrollWheelZoom: false, dragging: false,
  doubleClickZoom: false, touchZoom: false,
});
L.tileLayer(TILE_URL, TILE_OPTS).addTo(mapA);

var currentFrame = 0;
var heatLayer = L.heatLayer(HEAT_FRAMES[0], {
  radius: 11, blur: 11, maxZoom: 15, minOpacity: 0.15, max: 1.0,
  gradient: {
    0.00: '#0d1b2a',
    0.20: '#1a3a5c',
    0.45: '#1d6a8a',
    0.65: '#c47a2b',
    0.85: '#b84c2b',
    1.00: '#e8d5b0',
  },
}).addTo(mapA);

document.getElementById('decade-ghost').textContent = TIME_LABELS[0];

// ── Map B ─────────────────────────────────────────────────────
var mapB = L.map('map-b', {
  center: MAP_CENTER, zoom: MAP_ZOOM,
  zoomControl: false, attributionControl: false,
  scrollWheelZoom: false, dragging: false,
  doubleClickZoom: false, touchZoom: false,
});
L.tileLayer(TILE_URL, TILE_OPTS).addTo(mapB);

// All polygons start fully transparent — they fade in during transition
// so the choropleth appears to "crystallise" out of the heatmap glow
var geoJsonLayer = L.geoJson(GEOJSON, {
  style: function(feature) {
    return {
      fillColor:   'transparent',
      fillOpacity: 0,
      color:       'transparent',
      weight:      0,
    };
  },
  onEachFeature: function(feature, layer) {
    var name  = feature.properties[NAME_FIELD];
    var count = feature.properties['film_count'] || 0;
    layer.bindTooltip(
      '<b>' + name + '</b><br>' +
      '<span style="color:#ccc;font-size:11px;">' + count + ' filming locations</span>',
      { sticky: false, className: 'dark-tooltip' }
    );
    // Store target style for animation
    layer._targetStyle = FEATURE_STYLES[name] || {
      fillColor: '#0a0f1a', fillOpacity: 0.75,
      color: '#2a2a3a', weight: 0.5,
    };
  },
}).addTo(mapB);

// ── Choropleth fade-in animation ──────────────────────────────
// Animates each polygon from transparent to its target style
// over `duration` ms using requestAnimationFrame
function animateChoropleth(duration) {
  var start = null;

  // Collect all layers with a target style
  var layers = [];
  geoJsonLayer.eachLayer(function(layer) {
    if (layer._targetStyle) layers.push(layer);
  });

  function step(timestamp) {
    if (!start) start = timestamp;
    var progress = Math.min((timestamp - start) / duration, 1);
    // Ease out cubic: fast start, gentle finish
    var eased = 1 - Math.pow(1 - progress, 3);

    layers.forEach(function(layer) {
      var t = layer._targetStyle;
      layer.setStyle({
        fillColor:   t.fillColor,
        fillOpacity: t.fillOpacity * eased,
        color:       t.color,
        weight:      t.weight * eased,
        opacity:     eased,
      });
    });

    if (progress < 1) {
      requestAnimationFrame(step);
    }
  }
  requestAnimationFrame(step);
}

// Reverse: fade polygons back to invisible
function animateChoroplethOut(duration, callback) {
  var start  = null;
  var layers = [];
  geoJsonLayer.eachLayer(function(layer) {
    if (layer._targetStyle) layers.push(layer);
  });

  // Snapshot current opacity so we fade from wherever we are
  var snapshots = layers.map(function(layer) {
    return {
      fillOpacity: layer.options.fillOpacity || 0,
      weight:      layer.options.weight      || 0,
      opacity:     layer.options.opacity     || 0,
    };
  });

  function step(timestamp) {
    if (!start) start = timestamp;
    var progress = Math.min((timestamp - start) / duration, 1);
    var eased    = 1 - Math.pow(1 - progress, 3);
    var inv      = 1 - eased;

    layers.forEach(function(layer, i) {
      var s = snapshots[i];
      layer.setStyle({
        fillOpacity: s.fillOpacity * inv,
        weight:      s.weight      * inv,
        opacity:     s.opacity     * inv,
      });
    });

    if (progress < 1) {
      requestAnimationFrame(step);
    } else if (callback) {
      callback();
    }
  }
  requestAnimationFrame(step);
}

// ── Label overlay ─────────────────────────────────────────────
var labelPane      = mapB.getPane('overlayPane');
var labelContainer = document.createElement('div');
labelContainer.style.cssText = [
  'position:absolute', 'top:0', 'left:0',
  'width:100%', 'height:100%',
  'pointer-events:none', 'z-index:400',
  'opacity:0',                          // hidden until transition
  'transition:opacity 0.8s ease',
].join(';');
labelPane.appendChild(labelContainer);

var labelDivs = LABEL_DATA.map(function(d) {
  var el = document.createElement('div');
  el.textContent = d.label;
  el.style.cssText = [
    'position:absolute',
    'font-family:Helvetica Neue,sans-serif',
    'font-size:' + d.fontSize + 'px',
    'font-weight:700',
    'color:' + d.color,
    'opacity:' + d.opacity,
    'text-shadow:0 1px 3px rgba(0,0,0,0.7)',
    'white-space:nowrap',
    'transform:translate(-50%,-50%)',
    'will-change:left,top',
  ].join(';');
  labelContainer.appendChild(el);
  return el;
});

function updateLabels() {
  var origin = mapB.getPixelOrigin();
  LABEL_DATA.forEach(function(d, i) {
    var pt = mapB.project(L.latLng(d.lat, d.lng));
    labelDivs[i].style.left = (pt.x - origin.x) + 'px';
    labelDivs[i].style.top  = (pt.y - origin.y) + 'px';
  });
}
updateLabels();
mapB.on('moveend zoomend viewreset move', updateLabels);

// ── Transition state ──────────────────────────────────────────
var onMapB         = false;
var transitioning  = false;
var scrollCooldown = false;

function showMapB() {
  if (transitioning) return;
  transitioning = true;
  onMapB = true;

  // Step 1: cross-fade map containers (tiles are now identical so
  // the switch looks like nothing happened at the tile level)
  document.getElementById('map-b').style.opacity       = '1';
  document.getElementById('map-b').style.pointerEvents = 'auto';

  // Slight delay before fading map A out — lets both tile layers
  // sit on top of each other for one beat so there is no dark flash
  setTimeout(function() {
    document.getElementById('map-a').style.opacity        = '0';
    document.getElementById('decade-ghost').style.opacity = '0';
    document.getElementById('scroll-hint').style.opacity  = '0';
    document.getElementById('title-a').style.opacity      = '0';
  }, 200);

  // Step 2: animate the choropleth fill appearing (starts 400ms in)
  setTimeout(function() {
    animateChoropleth(1200);
  }, 400);

  // Step 3: fade in labels and UI after choropleth is mostly visible
  setTimeout(function() {
    labelContainer.style.opacity              = '1';
    document.getElementById('title-b').style.opacity  = '1';
    document.getElementById('legend-b').style.opacity = '1';
    mapB.invalidateSize();
    updateLabels();
    transitioning = false;
  }, 1400);
}

function showMapA() {
  if (transitioning) return;
  transitioning = true;
  onMapB = false;

  // Step 1: hide map B UI immediately
  labelContainer.style.opacity              = '0';
  document.getElementById('title-b').style.opacity  = '0';
  document.getElementById('legend-b').style.opacity = '0';
  document.getElementById('map-b').style.pointerEvents = 'none';

  // Step 2: animate choropleth fading out while map A fades back in
  animateChoroplethOut(600, null);

  setTimeout(function() {
    document.getElementById('map-a').style.opacity        = '1';
    document.getElementById('decade-ghost').style.opacity = '1';
    document.getElementById('title-a').style.opacity      = '1';
    document.getElementById('scroll-hint').style.opacity  = '1';
  }, 100);

  // Step 3: fade out map B container after map A is back
  setTimeout(function() {
    document.getElementById('map-b').style.opacity = '0';
    heatLayer.setLatLngs(HEAT_FRAMES[currentFrame]);
    document.getElementById('decade-ghost').textContent = TIME_LABELS[currentFrame];
    mapA.invalidateSize();
    transitioning = false;
  }, 700);
}

// ── Frame advance ─────────────────────────────────────────────
function advanceFrame(direction) {
  if (transitioning || scrollCooldown) return;
  scrollCooldown = true;
  setTimeout(function() { scrollCooldown = false; }, 340);

  if (onMapB) {
    if (direction < 0) showMapA();
    return;
  }

  if (direction > 0) {
    if (currentFrame < HEAT_FRAMES.length - 1) {
      currentFrame++;
      heatLayer.setLatLngs(HEAT_FRAMES[currentFrame]);
      document.getElementById('decade-ghost').textContent = TIME_LABELS[currentFrame];
    } else {
      showMapB();
    }
  } else {
    if (currentFrame > 0) {
      currentFrame--;
      heatLayer.setLatLngs(HEAT_FRAMES[currentFrame]);
      document.getElementById('decade-ghost').textContent = TIME_LABELS[currentFrame];
    }
  }
}

// ── Input listeners ───────────────────────────────────────────
window.addEventListener('wheel', function(e) {
  e.preventDefault();
  advanceFrame(e.deltaY > 0 ? 1 : -1);
}, { passive: false });

var touchStartY = 0;
window.addEventListener('touchstart', function(e) {
  touchStartY = e.touches[0].clientY;
}, { passive: true });
window.addEventListener('touchmove', function(e) {
  e.preventDefault();
  var delta = touchStartY - e.touches[0].clientY;
  if (Math.abs(delta) > 30) {
    advanceFrame(delta > 0 ? 1 : -1);
    touchStartY = e.touches[0].clientY;
  }
}, { passive: false });
window.addEventListener('keydown', function(e) {
  if (e.key === 'ArrowDown' || e.key === 'ArrowRight') advanceFrame(1);
  if (e.key === 'ArrowUp'   || e.key === 'ArrowLeft')  advanceFrame(-1);
});

</script>
</body>
</html>
"""

# Inject Python data
html_js = (html_js
    .replace('__HEAT_FRAMES__',    heat_frames_js)
    .replace('__TIME_LABELS__',    time_labels_js)
    .replace('__GEOJSON__',        geojson_js)
    .replace('__FEATURE_STYLES__', feature_styles_js)
    .replace('__LABEL_DATA__',     label_data_js)
    .replace('__NAME_FIELD__',     name_field_js)
)

final_html = html_head + html_js

with open("sf_scroll_experience.html", "w", encoding="utf-8") as f:
    f.write(final_html)

print("Saved sf_scroll_experience.html")
print(f"Decades: {time_labels}")
print(f"Scroll steps to transition: {len(time_labels)}")

Saved sf_scroll_experience.html
Decades: ['1910s', '1920s', '1930s', '1940s', '1950s', '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']
Scroll steps to transition: 12
